# FastAPI Interview Preparation (2026–27)
## Basic to Advanced | Python Backend / AI Engineer Fresher

**How to use this notebook**
1. Read each question and try answering aloud before opening the answer.
2. Run the code cells in order. For API examples, install the packages and start the server from a terminal.
3. Test endpoints using Swagger UI at `http://127.0.0.1:8000/docs` or Postman.
4. Change the sample data and practise explaining the result.

> The questions cover durable FastAPI/backend fundamentals commonly relevant to 2026–27 interviews. Actual company interview topics vary by role and employer.


## 0. Setup

Create a project folder and virtual environment. In Windows PowerShell:

```powershell
mkdir fastapi_interview
cd fastapi_interview
python -m venv .venv
.venv\Scripts\Activate.ps1
pip install fastapi uvicorn[standard] pydantic[email] pytest httpx
```

Create a file named `main.py` for the API examples. The notebook contains runnable Python examples as well as code to copy into `main.py`.


In [ ]:
# Check installed versions from this notebook
import sys
print("Python:", sys.version)

try:
    import fastapi, pydantic
    print("FastAPI:", fastapi.__version__)
    print("Pydantic:", pydantic.__version__)
except ImportError:
    print("Install packages using the setup commands in the previous cell.")


# Part 1 — Basic FastAPI concepts

### Q1. What is FastAPI?
**Answer:** FastAPI is a modern Python framework for building APIs. It uses Python type hints for validation and editor support, supports asynchronous endpoints, and generates interactive API documentation.

### Q2. What is an API?
**Answer:** An API (Application Programming Interface) is a defined way for software systems to communicate. For example, a mobile app can call a FastAPI endpoint to retrieve employee records.

### Q3. What is an endpoint?
**Answer:** An endpoint is an HTTP method and URL path that performs an operation, such as `GET /employees`.

### Q4. What is ASGI? What is Uvicorn?
**Answer:** ASGI is the asynchronous server interface standard for Python web applications. Uvicorn is an ASGI server that can run a FastAPI application.

### Q5. How do you start a FastAPI application?
**Answer:** If the file is `main.py` and the FastAPI instance is named `app`, run:
```powershell
uvicorn main:app --reload
```
`--reload` is useful during local development; it is generally not used as a production deployment setting.

### Q6. What is Swagger UI?
**Answer:** FastAPI provides interactive API documentation, usually at `/docs`. The alternative ReDoc page is usually at `/redoc`.

### Q7. What is the difference between Flask and FastAPI?
**Answer:** Both are Python web frameworks. FastAPI has first-class type-hint-based request validation, OpenAPI documentation generation, and ASGI/async support. Flask is a flexible WSGI framework with a smaller built-in API-specific feature set; extensions can add functionality.

### Q8. What are HTTP methods?
**Answer:** Common methods are GET (read), POST (create/submit), PUT (replace/update), PATCH (partial update), and DELETE (remove). Their exact semantics matter; for example, PUT is generally idempotent while POST is not necessarily idempotent.


# Part 2 — First runnable API

Save the following as `main.py`, then run `uvicorn main:app --reload` in the terminal.


In [ ]:
from fastapi import FastAPI

app = FastAPI(title="Interview Practice API")

@app.get("/")
def home():
    return {"message": "Welcome to FastAPI"}

@app.get("/hello/{name}")
def say_hello(name: str):
    return {"message": f"Hello, {name}"}

@app.get("/add")
def add_numbers(a: int, b: int):
    return {"result": a + b}


### Q9. What is the difference between path and query parameters?
**Answer:** A path parameter is part of the route, such as `/hello/Pratik`. A query parameter follows `?`, such as `/add?a=10&b=20`. Path parameters commonly identify a resource; query parameters commonly filter, sort, or configure a request.

### Q10. What happens if a query parameter declared as `int` receives text?
**Answer:** FastAPI validates the input and returns a 422 validation response rather than passing invalid text to the endpoint function.

### Q11. What is the difference between synchronous and asynchronous endpoints?
**Answer:** A normal `def` endpoint is synchronous. An `async def` endpoint can `await` asynchronous operations, allowing the event loop to serve other work while it waits. `async` does not automatically make CPU-heavy work faster. Use async-compatible libraries for I/O when appropriate.


# Part 3 — Pydantic, request bodies and validation

### Q12. What is Pydantic?
**Answer:** Pydantic validates and parses data using Python type annotations. FastAPI commonly uses Pydantic models to validate JSON request bodies.

### Q13. What is `BaseModel`?
**Answer:** It is the base class used to define Pydantic models.

### Q14. What is `Field()` used for?
**Answer:** `Field()` adds constraints and metadata, such as minimum length, maximum value, and descriptions.

### Q15. What is `EmailStr`?
**Answer:** It is a Pydantic email field type that validates email address format. Install its optional dependency with `pip install pydantic[email]`.

### Q16. What happens if a required field is missing?
**Answer:** Pydantic validation fails and FastAPI returns a 422 response containing validation details.

### Q17. What is `model_dump()`?
**Answer:** In Pydantic v2, `model_dump()` converts a model instance into a Python dictionary. Use `model_dump_json()` when you need a JSON string.

### Q18. What is `Literal`?
**Answer:** `Literal` restricts a field to specific allowed values, such as `"IT"`, `"HR"`, or `"Finance"`.

### Q19. What is the difference between request model and response model?
**Answer:** A request model validates incoming data. A response model validates and filters outgoing data, helping prevent accidental exposure of internal fields.


In [ ]:
from typing import Annotated, Literal
from pydantic import BaseModel, EmailStr, Field

class EmployeeCreate(BaseModel):
    name: Annotated[str, Field(min_length=2, max_length=50)]
    email: EmailStr
    department: Literal["IT", "HR", "Finance", "Sales", "Marketing"]
    salary: Annotated[int, Field(gt=0, le=5_000_000)]
    experience: Annotated[int, Field(ge=0, le=40)]
    status: Literal["Active", "Inactive"] = "Active"

class EmployeeResponse(BaseModel):
    employee_id: int
    name: str
    email: EmailStr
    department: str
    salary: int
    experience: int
    status: str

sample = EmployeeCreate(
    name="Pratik Shelke",
    email="pratik@example.com",
    department="IT",
    salary=600000,
    experience=1
)
print(sample.model_dump())


# Part 4 — CRUD API with in-memory data

This complete example provides create, read, update and delete operations. It stores data in memory, so records reset when the server restarts. It is for learning only; production applications should use a database and appropriate authentication/authorization.


In [ ]:
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, EmailStr, Field
from typing import Annotated, Literal

app = FastAPI(title="Employee CRUD API")

class EmployeeCreate(BaseModel):
    name: Annotated[str, Field(min_length=2, max_length=50)]
    email: EmailStr
    department: Literal["IT", "HR", "Finance", "Sales", "Marketing"]
    salary: Annotated[int, Field(gt=0)]
    experience: Annotated[int, Field(ge=0, le=40)]

class EmployeeUpdate(BaseModel):
    name: Annotated[str, Field(min_length=2, max_length=50)] | None = None
    department: Literal["IT", "HR", "Finance", "Sales", "Marketing"] | None = None
    salary: Annotated[int, Field(gt=0)] | None = None
    experience: Annotated[int, Field(ge=0, le=40)] | None = None

class EmployeeResponse(BaseModel):
    employee_id: int
    name: str
    email: EmailStr
    department: str
    salary: int
    experience: int

employees = {}
next_id = 1

@app.post("/employees", response_model=EmployeeResponse, status_code=status.HTTP_201_CREATED)
def create_employee(payload: EmployeeCreate):
    global next_id
    record = {"employee_id": next_id, **payload.model_dump()}
    employees[next_id] = record
    next_id += 1
    return record

@app.get("/employees", response_model=list[EmployeeResponse])
def list_employees(department: str | None = None):
    records = list(employees.values())
    if department:
        records = [e for e in records if e["department"].lower() == department.lower()]
    return records

@app.get("/employees/{employee_id}", response_model=EmployeeResponse)
def get_employee(employee_id: int):
    employee = employees.get(employee_id)
    if employee is None:
        raise HTTPException(status_code=404, detail="Employee not found")
    return employee

@app.put("/employees/{employee_id}", response_model=EmployeeResponse)
def update_employee(employee_id: int, payload: EmployeeUpdate):
    employee = employees.get(employee_id)
    if employee is None:
        raise HTTPException(status_code=404, detail="Employee not found")
    changes = payload.model_dump(exclude_unset=True, exclude_none=True)
    employee.update(changes)
    return employee

@app.delete("/employees/{employee_id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_employee(employee_id: int):
    if employee_id not in employees:
        raise HTTPException(status_code=404, detail="Employee not found")
    del employees[employee_id]
    return None


### Q20. What is CRUD?
**Answer:** Create, Read, Update and Delete. Commonly mapped to POST, GET, PUT/PATCH and DELETE.

### Q21. What is `HTTPException`?
**Answer:** It raises an HTTP error response. Example: `raise HTTPException(status_code=404, detail="Employee not found")`.

### Q22. What is `status_code=201`?
**Answer:** HTTP 201 Created indicates that a resource was successfully created. HTTP 200 means OK; HTTP 204 means success with no response body; HTTP 400 is a bad request; HTTP 401 is unauthenticated; HTTP 403 is forbidden; HTTP 404 is not found; HTTP 422 is validation failure; HTTP 500 is an internal server error.

### Q23. What is `response_model`?
**Answer:** It documents, validates and filters the returned data according to the declared Pydantic model. Do not return secrets such as password hashes in public response models.

### Q24. PUT vs PATCH?
**Answer:** PUT generally replaces a resource representation. PATCH applies a partial modification. In the example, PATCH-style behavior is implemented using `exclude_unset=True`.

### Q25. What does `exclude_unset=True` do?
**Answer:** It omits fields the client did not provide, allowing a partial update without overwriting existing values with defaults.


# Part 5 — JSON file storage

### Q26. How do you read and write JSON in Python?
**Answer:** `json.load(file)` reads JSON from an open file. `json.dump(data, file, indent=4)` writes Python data as formatted JSON.

### Q27. What is the difference between `json.load()` and `json.loads()`?
**Answer:** `load()` reads from a file-like object; `loads()` parses a JSON string.

### Q28. What should you consider before using a JSON file as a database?
**Answer:** File storage is simple for practice but lacks database transactions, robust concurrent writes, indexing, relational constraints and scalable querying. Prefer PostgreSQL/MySQL or another suitable database for production.


In [ ]:
import json
from pathlib import Path

DATA_FILE = Path("employees.json")

def load_employees():
    if not DATA_FILE.exists():
        return {}
    with DATA_FILE.open("r", encoding="utf-8") as file:
        return json.load(file)

def save_employees(data):
    with DATA_FILE.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=4)

# Example use
data = load_employees()
data["101"] = {
    "name": "Pratik Shelke",
    "department": "IT"
}
save_employees(data)
print(load_employees())


# Part 6 — Intermediate and advanced topics

### Q29. What is dependency injection in FastAPI?
**Answer:** Dependency injection lets an endpoint declare the services or resources it needs. FastAPI resolves them using `Depends()`. It is useful for shared logic, authentication, database sessions and configuration.

### Q30. What is middleware?
**Answer:** Middleware is code that runs around request/response processing. It can be used for request logging, timing, CORS handling and other cross-cutting concerns.

### Q31. What is CORS?
**Answer:** Cross-Origin Resource Sharing is a browser security mechanism that controls whether a web page from one origin can access resources from another origin. Configure only the origins and methods your frontend needs.

### Q32. What is authentication vs authorization?
**Answer:** Authentication verifies who the user is. Authorization determines what that authenticated user is allowed to do.

### Q33. What is OAuth2?
**Answer:** OAuth2 is an authorization framework commonly used to grant limited access using tokens. FastAPI provides security utilities that help implement OAuth2 flows; a real application still needs secure token issuance, validation and authorization rules.

### Q34. What is JWT?
**Answer:** JSON Web Token is a compact token format that can carry signed claims. A signed JWT is not automatically encrypted. Validate signature, expiry, issuer, audience and relevant claims; do not put secrets in its payload.

### Q35. What is `Depends()`?
**Answer:** `Depends()` declares a dependency that FastAPI should provide to an endpoint or another dependency.

### Q36. How do you connect FastAPI to a database?
**Answer:** Use a database driver or ORM such as SQLAlchemy with a suitable database. Manage sessions/connections safely, use migrations, parameterized queries, and transactions.

### Q37. What is the difference between `def` and `async def` for database calls?
**Answer:** Use `async def` with an async database driver/session when the operation is asynchronous. A blocking synchronous database call inside an async endpoint can block the event loop. A synchronous endpoint can be appropriate with a synchronous driver.

### Q38. What is `BackgroundTasks`?
**Answer:** It schedules lightweight work to run after the response is sent, such as sending a simple notification. For durable, long-running or high-volume jobs, use a task queue such as Celery or another worker system.

### Q39. What are lifespan events?
**Answer:** Lifespan handlers manage application startup and shutdown resources, such as initializing or closing a shared client or connection pool. They are preferred over older startup/shutdown event patterns for modern FastAPI apps.

### Q40. How do you test a FastAPI application?
**Answer:** Use `TestClient` with pytest for endpoint tests. Test success cases, validation failures, missing resources, permissions and edge cases. Mock external services where appropriate.


In [ ]:
# Minimal FastAPI test example (put API code in main.py first)
# Run in terminal: pytest -q

# Save as test_main.py:
#
# from fastapi.testclient import TestClient
# from main import app
#
# client = TestClient(app)
#
# def test_home():
#     response = client.get("/")
#     assert response.status_code == 200
#
# def test_create_employee():
#     response = client.post("/employees", json={
#         "name": "Test User",
#         "email": "test@example.com",
#         "department": "IT",
#         "salary": 50000,
#         "experience": 1
#     })
#     assert response.status_code == 201
#     assert response.json()["name"] == "Test User"


# Part 7 — Practical coding-round tasks with solutions

Try each task yourself before reading the solution.

## Task 1: Create a GET endpoint that returns a welcome message
**Expected:** `{"message": "Welcome to Employee API"}`

Solution:
```python
@app.get("/welcome")
def welcome():
    return {"message": "Welcome to Employee API"}
```

## Task 2: Create a path parameter for a product ID
Solution:
```python
@app.get("/products/{product_id}")
def get_product(product_id: int):
    return {"product_id": product_id}
```

## Task 3: Filter employees by department
Solution:
```python
@app.get("/employees/by-department/{department}")
def filter_department(department: str):
    result = [
        employee for employee in employees.values()
        if employee["department"].casefold() == department.casefold()
    ]
    return result
```

## Task 4: Add an employee with validation
Solution:
```python
@app.post("/employees", status_code=201)
def add_employee(employee: EmployeeCreate):
    # In a real app, save this record in a database.
    return {"message": "Employee received", "employee": employee.model_dump()}
```

## Task 5: Return 404 if an employee does not exist
Solution:
```python
from fastapi import HTTPException

@app.get("/employees/{employee_id}")
def find_employee(employee_id: int):
    employee = employees.get(employee_id)
    if employee is None:
        raise HTTPException(status_code=404, detail="Employee not found")
    return employee
```

## Task 6: Add pagination
Solution:
```python
@app.get("/employees/page")
def get_employee_page(skip: int = 0, limit: int = 10):
    if skip < 0 or limit < 1 or limit > 100:
        raise HTTPException(status_code=400, detail="Invalid pagination values")
    records = list(employees.values())
    return records[skip:skip + limit]
```


# Part 8 — Scenario-based questions

### Q41. Your POST API returns 422. What do you check?
**Answer:** Check the request body format, required fields, field names, data types, `Literal` allowed values, validation constraints and the detailed error response.

### Q42. Your API returns 404. What do you check?
**Answer:** Confirm the route and HTTP method, check the requested ID exists, and verify the lookup uses the correct key type (for example, string vs integer).

### Q43. Your API returns 500. What do you do?
**Answer:** Read the server traceback, reproduce the request, inspect the failing line and input, check file/database availability, and fix the underlying exception. Avoid exposing internal stack traces to clients in production.

### Q44. Your endpoint is slow. How do you investigate?
**Answer:** Measure latency, inspect logs/traces, profile CPU time, examine database queries and external API calls, check blocking operations, and add caching or pagination only when measurements justify them.

### Q45. How would you secure an employee API?
**Answer:** Use HTTPS, authentication, role/permission checks, input validation, safe secret storage, rate limiting where needed, secure CORS configuration, audit logs and safe error messages. Never store plaintext passwords.

### Q46. How would you deploy FastAPI?
**Answer:** Package the app and dependencies, configure environment variables/secrets, run an ASGI server in a production setup, use HTTPS through a suitable proxy/load balancer, add health checks and logging, and monitor the service. Deployment details depend on the platform.


# Part 9 — Rapid revision: important status codes

| Code | Meaning | Example |
|---:|---|---|
| 200 | OK | Successful GET |
| 201 | Created | New employee created |
| 204 | No Content | Successful delete |
| 400 | Bad Request | Invalid request |
| 401 | Unauthorized | Missing/invalid authentication |
| 403 | Forbidden | User lacks permission |
| 404 | Not Found | Employee ID does not exist |
| 409 | Conflict | Duplicate resource |
| 422 | Unprocessable Content | Request validation failed |
| 500 | Internal Server Error | Unexpected server-side error |

# Part 10 — Interview self-practice checklist

- [ ] Explain FastAPI, ASGI and Uvicorn in your own words.
- [ ] Build GET, POST, PUT, PATCH and DELETE endpoints.
- [ ] Explain path parameters vs query parameters.
- [ ] Create Pydantic models with `Field`, `EmailStr` and `Literal`.
- [ ] Explain `model_dump()` and `response_model`.
- [ ] Handle 404 and validation errors.
- [ ] Read and write JSON safely.
- [ ] Explain dependency injection and middleware.
- [ ] Explain authentication, authorization, OAuth2 and JWT.
- [ ] Connect a database and write pytest API tests.
- [ ] Explain how you would deploy and monitor an API.

## Suggested study order
**Week 1:** Basic concepts, routing, HTTP methods, path/query parameters.  
**Week 2:** Pydantic, validation, CRUD, status codes, error handling.  
**Week 3:** JSON/database integration, dependency injection, middleware, testing.  
**Week 4:** Authentication, async behavior, deployment, scenario questions and mock interviews.

**Interview reminder:** Be ready to explain your own project code, why you chose each design, what errors you faced, and how you tested the API.
